## Assignment 1: Prompting for Program Repair

In this assignment, you will design prompts to repair buggy Python functions and evaluate the quality of the bug repair using unit tests.

What you will do:
- Use this notebook as a guide to complete your assignment.
- Inside the `data` directory, you’ll find one folder per buggy Python function (`data/BUGxxx/`), each with example tests in `test_bug.py`.
- Experiment with prompting techniques using the model (Gemma 3 270M IT), and observe how variations in prompt wording affect bug repair performance.
- For each buggy Python function, record the prompts you used, then choose one final repaired code to submit.

Submission Instructions:  
Submit a ZIP file of the `results` folder, containing the two items listed below, named `SWE270P_A1.ZIP`.  
**Please make sure to strictly follow the submission format and filename, as an autograder will be used for grading. <br> An incorrect format or filename may result in a score of 0.**  
 1) `prompts.jsonl`: all the prompts you used. This notebook includes a helper function that logs them to `results/prompts.jsonl`.
 2) `BUGxxx.py` files: one final repaired code file per buggy Python function (e.g., `BUG001.py`), saved under `results/final_patches/`.  


Required File Structure:  
Your submission ZIP file **must** follow this exact file structure for compatibility with the autograder:    
```
SWE270P_A1.ZIP
└── results
    ├── prompts.jsonl
    └── final_patches
        ├── BUG000.py
        ├── ...
        └── BUG009.py
```

Helpful Guidelines:
- Use `test_bug.py` to evaluate the quality of repaired code.
- Add your own test cases to check the robustness of your repaired code.
- Final submissions will be graded using an autograder. Do not modify any function names or parameter definitions.
- Each final patch must be a complete replacement for that bug's `buggy.py`. Keep its imports and helper functions, not only the fixed function (e.g., BUG001 needs `get` and `insert_or_update`).

Recommended process:
1) Work on one bug at a time (e.g., BUG001).
2) Select a prompting technique (e.g., zero-shot, chain-of-thought, tests-aware, or self-refine).
3) Using the selected technique, draft your prompt and call the model.
4) Copy the repaired code into that bug's `buggy.py` and run its tests with `test_bug.py` (e.g., in `data/BUG001/`). <br>
    **For example**, for BUG001, run `pytest test_bug.py` in a terminal inside `data/BUG001/`, or run `!cd data/BUG001 && pytest test_bug.py` in a notebook cell.
5) Modify the prompt using different wording and structure, or try another prompting technique until the repaired code passes all tests.   
6) Log the prompting technique and the prompt used to call the model in `prompts.jsonl`.
7) Save your final repaired code for each bug (e.g., `BUG001.py`) for submission.



## 0. Setup (Required: Gemma 3 270M IT)

This assignment **must use** `google/gemma-3-270m-it` via Hugging Face Transformers (local CPU/GPU). No external APIs are needed.

We recommend using Google Colab. If you're using Colab, make sure you have a GPU runtime (Runtime → Change runtime type → select T4 GPU). If you run locally, use Python 3.10 or newer.

In [ ]:
# Run this only if you are running locally
%pip install transformers ipywidgets torch accelerate

In [ ]:
%pip install pytest huggingface_hub

Note:

1. To use Hugging Face models, you need a Hugging Face access token (see [User access tokens](https://huggingface.co/docs/hub/en/security-tokens)).  
A **Read** token works. If you create a fine-grained token instead, check the box for "Read access to contents of all public gated repos you can access". Paste the token into the code below when prompted.  

2. Gemma 3 is a gated model, so you need to request access on the [model page](https://huggingface.co/google/gemma-3-270m-it) before downloading it.

In [ ]:
from huggingface_hub import login
login()

In [ ]:
# Load model
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-270m-it")
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-270m-it",
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

def generate_text(prompt: str, max_new_tokens: int = 200) -> str:
    # TODO: Write a function to call the gemma-3-270m-it model for inference
    # Refer to the model documentation for more details: https://huggingface.co/google/gemma-3-270m-it
    # Hint: format the prompt with tokenizer.apply_chat_template(...), then decode only the newly generated
    # tokens with skip_special_tokens=True so the output doesn't end with <end_of_turn>.
    pass

# Check if the model is running (uncomment the print statement below):
# print(generate_text("Hello from Gemma-3-it!"))


## Prompting Techniques: Zero‑Shot, One‑Shot, Chain‑of‑Thought, and Few‑Shot  
**Goal:** Practice different prompting strategies and compare their effects on bug repair correctness.

After generating the fix for a problem, you can paste it into the corresponding `buggy.py` and run `pytest` for that folder to see if it passes.

In [ ]:
from pathlib import Path
import json, re, datetime as dt
from typing import Optional

"""
Helper function to save the prompts.jsonl file.
TODO: Write a helper function to extract the repaired code from the LLM response.
"""

PROMPTS_PATH = Path("results/prompts.jsonl")
PATCH_DIR = Path("results/final_patches")                        # Use this folder to save the repaired Python code
PROMPTS_PATH.parent.mkdir(parents=True, exist_ok=True)
PATCH_DIR.mkdir(parents=True, exist_ok=True)

def log_prompt(technique: str, prompt: str,
               completion: Optional[str] = None,
               bug_file: Optional[str] = None,
               model: Optional[str] = None,
               ):
    rec = {
        "timestamp": dt.datetime.now().isoformat(timespec="seconds"),
        "technique": technique,
        "prompt": prompt,
    }
    if completion is not None: rec["completion"] = completion
    if bug_file is not None:   rec["bug_file"] = bug_file
    if model is not None:      rec["model"] = model
    with PROMPTS_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def save_final_patch(bug_file: str, model_output: str) -> str:
    """bug_file is a file name like 'BUG001.py'. Saves to results/final_patches/BUG001.py

    Save the complete file (imports and helper functions included), not only the fixed function.
    """
    # TODO: Write your own helper function to extract repaired code from LLM-generated responses.
    pass


## 1. Zero‑Shot
A prompting technique that asks the model to solve the task **without** examples. Include specific output constraints.

In [ ]:
example_zero_shot_prompt = """You are a helpful coding assistant.

Fix the bug in this function.
Function:
```python
def add_one(x):
    return x - 1
```
Requirement: Make it return x + 1.
Output: Return only the corrected function in a Python code block, nothing else.
"""

print("----- ZERO-SHOT EXAMPLE PROMPT -----\n", example_zero_shot_prompt)
example_zero_shot_output = generate_text(example_zero_shot_prompt, max_new_tokens=120)
print("\n----- ZERO-SHOT EXAMPLE OUTPUT -----\n", example_zero_shot_output[:1000])


In [ ]:
# For example, your function calls should save the log and the final patch as shown below.
# You can use this example code to save the log and the final patched Python file for your submission.
# Note: Make sure to use the correct file name by replacing BUG_xxx.py with BUGxxx.py for your submission.
log_prompt("zero-shot", example_zero_shot_prompt, completion=example_zero_shot_output, bug_file="BUG_xxx.py", model="google/gemma-3-270m-it")
save_final_patch("BUG_xxx.py", example_zero_shot_output)

In [ ]:
# Your turn: write a zero-shot prompt below
your_zero_shot_prompt = """
"""

print("\n----- YOUR ZERO-SHOT PROMPT -----\n", your_zero_shot_prompt)
your_zero_shot_output = generate_text(your_zero_shot_prompt, max_new_tokens=200)
print("\n----- YOUR ZERO-SHOT OUTPUT -----\n", your_zero_shot_output[:1000])

## 2. One‑Shot
A prompting technique that includes **one worked example** before the target task.

In [ ]:
example_one_shot_prompt = """You are a helpful coding assistant.

Example (Problem → Solution)
Problem: Write a Python function add_one(x) that returns x + 1.
Solution:
```python
def add_one(x):
    return x + 1
```

Now fix a new function (One-Shot):
Function:
```python
def mul_two(a, b):
    return a + b
```
Requirement: Make it return a * b.
Output: Return only the corrected function in a Python code block.
"""

print("----- ONE-SHOT EXAMPLE PROMPT -----\n", example_one_shot_prompt)
example_one_shot_output = generate_text(example_one_shot_prompt, max_new_tokens=160)
print("\n----- ONE-SHOT EXAMPLE OUTPUT -----\n", example_one_shot_output[:1000])



In [ ]:
# Your turn: write a one-shot prompt below
your_one_shot_prompt = """
"""

print("\n----- YOUR ONE-SHOT PROMPT -----\n", your_one_shot_prompt)
your_one_shot_output = generate_text(your_one_shot_prompt, max_new_tokens=220)
print("\n----- YOUR ONE-SHOT OUTPUT -----\n", your_one_shot_output[:1000])

## 3. Chain‑of‑Thought (CoT)
A prompting technique that asks the model for a brief high‑level plan (step-by-step) before the final code.

In [ ]:
cot_example_prompt = """You are a helpful coding assistant.

Task: Fix the bug in this function.
Function:
```python
def is_even(n):
    return n % 2 == 1
```

Think step by step in 2-4 short bullet points about the bug and the fix.
After your reasoning, output ONLY the final corrected function in a Python code block and nothing else.
"""

print("----- COT EXAMPLE PROMPT -----\n", cot_example_prompt)
cot_example_output = generate_text(cot_example_prompt, max_new_tokens=220)
print("\n----- COT EXAMPLE OUTPUT -----\n", cot_example_output[:1000])



In [ ]:
# Your turn: write a CoT prompt below
your_cot_prompt = """
"""

print("\n----- YOUR COT PROMPT -----\n", your_cot_prompt)
your_cot_output = generate_text(your_cot_prompt, max_new_tokens=240)
print("\n----- YOUR COT OUTPUT -----\n", your_cot_output[:1000])

## 4. Few‑Shot
A prompting technique that provides multiple examples to shape the model's behavior.

In [ ]:
few_shot_example_prompt = """You are a helpful coding assistant.

Here are two examples (Problem → Solution):

Problem: Write a function mul_two(a, b) that returns a * b.
Solution:
```python
def mul_two(a, b):
    return a * b
```

Problem: Return the larger of two numbers.
Solution:
```python
def max_of_two(a, b):
    return a if a >= b else b
```

Now fix a new function:
Function:
```python
def square(x):
    return x + x
```
Requirement: Make it return x * x.
Output: Return only the corrected function in a Python code block.
"""

print("----- FEW-SHOT EXAMPLE PROMPT -----\n", few_shot_example_prompt)
few_shot_example_output = generate_text(few_shot_example_prompt, max_new_tokens=200)
print("\n----- FEW-SHOT EXAMPLE OUTPUT -----\n", few_shot_example_output[:1000])



In [ ]:
# Your turn: write a few-shot prompt below
your_few_shot_prompt = """You are a helpful coding assistant.

Provide 1-3 tiny examples above the task (Problem → Solution), then present your target function to fix.

[Write your examples here as Problem/Solution code blocks.]

Now solve a new task:
Task: [paste or describe the buggy function]
Requirement: [expected behavior]
Output: Return only the corrected Python function in a single code block.
"""

print("\n----- YOUR FEW-SHOT PROMPT -----\n", your_few_shot_prompt)
your_few_shot_output = generate_text(your_few_shot_prompt, max_new_tokens=260)
print("\n----- YOUR FEW-SHOT OUTPUT -----\n", your_few_shot_output[:1000])

## Submission checklist

Before you submit:
- Ensure `results/prompts.jsonl` contains all prompts you tried (one JSON line per attempt). <br> FYI: Make sure to check the rubric for full marks on this!
- Ensure you have one final repaired code for each buggy Python function, saved under `results/final_patches/`.
- Make sure each file is named exactly `BUGxxx.py` (e.g., `BUG001.py`).

What to upload:
- `results/prompts.jsonl`
- `results/final_patches/` folder containing all 10 repaired code files, one per buggy Python function

**Required File Structure**:  
Your submission ZIP file **must** follow this exact file structure for compatibility with the autograder:    
```
SWE270P_A1.ZIP
└── results
    ├── prompts.jsonl
    └── final_patches
        ├── BUG000.py
        ├── ...
        └── BUG009.py
```